# Course 3 — LDA & QDA

Linear and Quadratic Discriminant Analysis: from Bayes theorem to curved decision boundaries.

**Sections**
1. Two philosophies: discriminative vs generative (0:00–0:15)
2. Bayes theorem for classification (0:15–0:30)
3. Gaussian LDA — model, discriminant scores, linear boundary (0:30–0:50)
4. Parameter estimation (0:50–1:00)
5. Demo: LDA on Default — confusion matrix, threshold table, ROC vs LR (1:00–1:15)
6. Fisher's linear discriminant projection on Iris (1:15–1:30)
7. QDA — separate Σₖ, quadratic boundary, bias-variance tradeoff (1:30–2:00)

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
while not (p / 'pyproject.toml').exists() and p != p.parent:
    p = p.parent
if str(p) not in sys.path:
    sys.path.insert(0, str(p))
from shared.data_utils import load_dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
                              roc_auc_score, classification_report)
from sklearn.preprocessing import LabelEncoder

default = load_dataset('default')
iris    = load_dataset('iris')
d = default.copy()
d['y'] = (d['default'] == 'Yes').astype(int)
y_bin = d['y'].to_numpy()
print('default:', d.shape, '  iris:', iris.shape)

## 1. Discriminative vs Generative classifiers

| Approach | What is modelled | Examples |
|---|---|---|
| **Discriminative** | P(Y=k \| X=x) directly | Logistic Regression |
| **Generative** | P(X=x \| Y=k) + prior π_k | LDA, QDA, Naive Bayes |

Generative models use Bayes theorem to recover P(Y=k|X=x) from the class-conditional density.
When the modelling assumptions hold, generative models can outperform discriminative ones —
especially with **small n** or when classes are **well separated** (where LR coefficients blow up).

### When to prefer LDA over Logistic Regression
- Well-separated classes (LR has numerical instability)
- Small n — LDA is more stable with fewer observations
- Features are approximately Gaussian
- Multiple outputs: LDA naturally gives a low-dim projection (Fisher's discriminant)

## 2. Bayes Theorem for Classification

$$\Pr(Y=k \mid X=x) = \frac{\pi_k \cdot f_k(x)}{\sum_{l=1}^{K} \pi_l \cdot f_l(x)}$$

- **π_k = Pr(Y=k)**: prior probability of class k
- **f_k(x) = p(X=x | Y=k)**: class-conditional density — this is what LDA models
- The denominator normalizes so probabilities sum to 1

The **Bayes classifier** assigns x to the class k with the highest posterior — it is optimal when the distributions are known exactly. LDA approximates these distributions with Gaussian densities.

In [ ]:
# Illustrate Bayes theorem: two 1D Gaussians with different means
x_range = np.linspace(-4, 8, 400)
mu0, mu1, sigma = 0.0, 4.0, 1.5
pi0, pi1 = 0.7, 0.3

from scipy.stats import norm
f0 = norm.pdf(x_range, mu0, sigma)
f1 = norm.pdf(x_range, mu1, sigma)
posterior1 = pi1 * f1 / (pi0 * f0 + pi1 * f1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Left: class-conditional densities
axes[0].fill_between(x_range, pi0 * f0, alpha=0.4, label=f'π₀·f₀(x), class 0 (π={pi0})')
axes[0].fill_between(x_range, pi1 * f1, alpha=0.4, color='C3', label=f'π₁·f₁(x), class 1 (π={pi1})')
axes[0].set_xlabel('x'); axes[0].set_ylabel('density × prior')
axes[0].set_title('Class-conditional densities weighted by prior')
axes[0].legend(fontsize=9)
# Right: posterior probability
axes[1].plot(x_range, posterior1, color='C3', linewidth=2)
axes[1].axhline(0.5, color='k', linestyle='--', alpha=0.4)
boundary = (mu0 + mu1) / 2 - sigma**2 * np.log(pi1/pi0) / (mu1 - mu0)
axes[1].axvline(boundary, color='gray', linestyle=':', label=f'Bayes boundary ≈ {boundary:.2f}')
axes[1].set_xlabel('x'); axes[1].set_ylabel('Pr(Y=1 | X=x)')
axes[1].set_title('Posterior probability of class 1')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()
print(f'Bayes decision boundary at x ≈ {boundary:.3f}')

## 3. Gaussian LDA — The Model

LDA assumes each class-conditional density is Gaussian with a **shared** covariance matrix Σ:

$$f_k(x) = \frac{1}{(2\pi)^{p/2}|\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(x-\mu_k)^\top \Sigma^{-1}(x-\mu_k)\right)$$

### Discriminant score derivation

Plugging into Bayes and taking the log (monotone → same argmax):

$$\delta_k(x) = x^\top \Sigma^{-1}\mu_k - \frac{1}{2}\mu_k^\top \Sigma^{-1}\mu_k + \log \pi_k$$

**Why linear?** Expand $(x-\mu_k)^\top \Sigma^{-1}(x-\mu_k) = x^\top\Sigma^{-1}x - 2\mu_k^\top\Sigma^{-1}x + \mu_k^\top\Sigma^{-1}\mu_k$.
The $x^\top\Sigma^{-1}x$ term is the **same for every class k** (shared Σ), so it cancels in comparisons. What remains is linear in x.

### Decision boundary between classes k and l

Set δ_k(x) = δ_l(x):
$$x^\top \Sigma^{-1}(\mu_k - \mu_l) = \frac{1}{2}(\mu_k^\top\Sigma^{-1}\mu_k - \mu_l^\top\Sigma^{-1}\mu_l) - \log(\pi_k/\pi_l)$$

Left side is **linear in x** → the boundary is a hyperplane (line in 2D).

## 4. Parameter Estimation

$$\hat{\pi}_k = \frac{n_k}{n}, \quad \hat{\mu}_k = \frac{1}{n_k}\sum_{i:\,y_i=k} x_i$$

$$\hat{\Sigma} = \frac{1}{n-K}\sum_{k=1}^{K}\sum_{i:\,y_i=k}(x_i - \hat{\mu}_k)(x_i - \hat{\mu}_k)^\top \quad \text{(pooled within-class covariance)}$$

These are just sample means and pooled sample covariance — simple closed-form estimates, no iterative optimization needed.

In [ ]:
# Compute LDA estimates by hand for intuition
X2 = d[['balance', 'income']].to_numpy()
classes = [0, 1]
n = len(y_bin)
pi_hat = {k: (y_bin == k).mean() for k in classes}
mu_hat  = {k: X2[y_bin == k].mean(axis=0) for k in classes}
# Pooled within-class covariance
S_pooled = sum((X2[y_bin == k] - mu_hat[k]).T @ (X2[y_bin == k] - mu_hat[k])
               for k in classes) / (n - len(classes))
print('Prior probabilities:')
for k in classes:
    print(f'  π̂_{k} = {pi_hat[k]:.4f}')
print('\nClass means (balance, income):')
for k in classes:
    print(f'  μ̂_{k} = {mu_hat[k].round(2)}')
print('\nPooled covariance matrix (2×2):')
print(pd.DataFrame(S_pooled, columns=['balance','income'],
                   index=['balance','income']).round(2))

## 5. LDA on Default Dataset

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X2, y_bin, test_size=0.3, random_state=0, stratify=y_bin)
lda = LinearDiscriminantAnalysis().fit(Xtr, ytr)
lr  = LogisticRegression(max_iter=2000).fit(Xtr, ytr)

print('LDA training accuracy:', lda.score(Xtr, ytr).round(4))
print('LDA held-out accuracy:', lda.score(Xte, yte).round(4))
print('LR  held-out accuracy:', lr.score(Xte, yte).round(4))

In [ ]:
# Confusion matrix at default threshold
pred_lda = lda.predict(Xte)
cm = confusion_matrix(yte, pred_lda)
tn, fp, fn, tp = cm.ravel()
print('Confusion matrix at threshold 0.5 (LDA on Default):')
print(pd.DataFrame(cm, index=['Actual: No Default', 'Actual: Default'],
                   columns=['Pred: No Default', 'Pred: Default']))
print(f'\nFPR = {fp/(fp+tn):.4f}  (false alarm rate)')
print(f'FNR = {fn/(fn+tp):.4f}  (missed defaulters — costly!)')
print(f'TPR = {tp/(tp+fn):.4f}  (recall of defaulters)')
print(f'Overall accuracy = {(tn+tp)/len(yte):.4f}')

In [ ]:
# Threshold sensitivity — recreates ISLR Table 4.4
proba_lda = lda.predict_proba(Xte)[:, 1]
print('Threshold sensitivity (LDA on Default):')
print(f'{"Threshold":>12} {"FPR":>8} {"FNR":>8} {"Overall Error":>15}')
for t in [0.05, 0.10, 0.20, 0.50]:
    pred_t = (proba_lda >= t).astype(int)
    tn_t, fp_t, fn_t, tp_t = confusion_matrix(yte, pred_t).ravel()
    fpr_t = fp_t / (fp_t + tn_t)
    fnr_t = fn_t / (fn_t + tp_t) if (fn_t + tp_t) > 0 else 0
    err_t = (fp_t + fn_t) / len(yte)
    print(f'{t:>12.2f} {fpr_t:>8.3f} {fnr_t:>8.3f} {err_t:>15.3f}')

In [ ]:
# ROC comparison: LDA vs Logistic Regression
proba_lr  = lr.predict_proba(Xte)[:, 1]
fpr_lda, tpr_lda, _ = roc_curve(yte, proba_lda)
fpr_lr,  tpr_lr,  _ = roc_curve(yte, proba_lr)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr_lda, tpr_lda, linewidth=2,
        label=f'LDA  (AUC = {auc(fpr_lda, tpr_lda):.4f})')
ax.plot(fpr_lr,  tpr_lr,  linewidth=2, linestyle='--',
        label=f'LR   (AUC = {auc(fpr_lr, tpr_lr):.4f})')
ax.plot([0,1], [0,1], 'k:', alpha=0.4, label='Random')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.set_title('ROC: LDA vs Logistic Regression (Default dataset)')
ax.legend(); plt.show()
print('With approximately Gaussian features, LDA ≈ LR in predictive performance.')

In [ ]:
# Visualize LDA decision boundary on Default (balance vs income)
x0_range = np.linspace(X2[:,0].min()-200, X2[:,0].max()+200, 300)
x1_range = np.linspace(X2[:,1].min()-5000, X2[:,1].max()+5000, 300)
xx, yy = np.meshgrid(x0_range, x1_range)
Z = lda.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdBu')
ax.scatter(X2[y_bin==0, 0], X2[y_bin==0, 1], s=3, alpha=0.3, label='No default')
ax.scatter(X2[y_bin==1, 0], X2[y_bin==1, 1], s=15, color='C3', alpha=0.7,
           edgecolors='k', linewidths=0.4, label='Default')
ax.set_xlabel('balance'); ax.set_ylabel('income')
ax.set_title('LDA decision boundary — Default dataset')
ax.legend(); plt.show()

## 6. Fisher's Linear Discriminant Projection

Fisher (1936) found the projection direction w that maximizes the ratio:

$$J(w) = \frac{w^\top S_B w}{w^\top S_W w}$$

where $S_B$ = between-class scatter matrix and $S_W$ = within-class scatter matrix.

**Solution**: $w = S_W^{-1}(\mu_1 - \mu_2)$ — same direction as LDA's discriminant!

With K classes and p features: there are **min(K−1, p)** discriminant axes.
This gives a principled low-dimensional projection for visualization and dimensionality reduction.

In [ ]:
# Fisher projection on 4D Iris → 2D (K-1 = 2 axes)
le = LabelEncoder().fit(iris['species'])
y_iris = le.transform(iris['species'])
X4 = iris[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']].to_numpy()

lda_iris = LinearDiscriminantAnalysis().fit(X4, y_iris)
X_proj = lda_iris.transform(X4)  # (150, 2) — Fisher projection

print(f'Explained variance ratio by LD1, LD2: {lda_iris.explained_variance_ratio_.round(4)}')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['C0', 'C1', 'C2']
# 2D Fisher projection
for cls, (name, col) in enumerate(zip(le.classes_, colors)):
    mask = y_iris == cls
    axes[0].scatter(X_proj[mask, 0], X_proj[mask, 1], s=25, label=name,
                    color=col, edgecolors='k', linewidths=0.4)
axes[0].set_xlabel('LD1'); axes[0].set_ylabel('LD2')
axes[0].set_title('Fisher (LDA) projection — Iris 4D → 2D')
axes[0].legend()
# 1D histogram along LD1
for cls, (name, col) in enumerate(zip(le.classes_, colors)):
    axes[1].hist(X_proj[y_iris == cls, 0], bins=20, alpha=0.6, label=name, color=col)
axes[1].set_xlabel('LD1 score'); axes[1].set_ylabel('count')
axes[1].set_title('Class histograms along LD1')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Discriminant coefficients: which features contribute most?
coef_df = pd.DataFrame(lda_iris.coef_,
                        index=le.classes_,
                        columns=['sepal_len','sepal_wid','petal_len','petal_wid'])
print('LDA discriminant coefficient matrix (one row per class):')
print(coef_df.round(4))
print('\nScaling (direction in feature space):')
scaling_df = pd.DataFrame(lda_iris.scalings_,
                           index=['sepal_len','sepal_wid','petal_len','petal_wid'],
                           columns=['LD1','LD2'])
print(scaling_df.round(4))

## 7. QDA — Quadratic Discriminant Analysis

QDA relaxes the shared-Σ assumption: each class gets its own covariance matrix Σ_k.

$$\delta_k(x) = -\frac{1}{2}\log|\Sigma_k| - \frac{1}{2}(x-\mu_k)^\top \Sigma_k^{-1}(x-\mu_k) + \log \pi_k$$

The term $x^\top \Sigma_k^{-1} x$ **no longer cancels** (Σ_k differs by class) → **quadratic boundary**.

### Parameter count

| Method | Covariance parameters | Boundary shape |
|---|---|---|
| LDA | p(p+1)/2 (shared) | Linear hyperplane |
| QDA | K · p(p+1)/2 (per class) | Quadratic surface |

With p=4, K=3: LDA has 10 Σ parameters; QDA has 30. Need more data.

In [ ]:
# LDA vs QDA on Iris (2 features for visualization)
X2_iris = iris[['petal_length', 'petal_width']].to_numpy()
lda2 = LinearDiscriminantAnalysis().fit(X2_iris, y_iris)
qda2 = QuadraticDiscriminantAnalysis().fit(X2_iris, y_iris)

x_min, x_max = X2_iris[:,0].min()-0.5, X2_iris[:,0].max()+0.5
y_min, y_max = X2_iris[:,1].min()-0.5, X2_iris[:,1].max()+0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                      np.linspace(y_min, y_max, 300))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, model, title in zip(axes, [lda2, qda2], ['LDA — linear boundary', 'QDA — quadratic boundary']):
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, levels=[-0.5,0.5,1.5,2.5], cmap='Set1')
    for cls, (name, col) in enumerate(zip(le.classes_, ['C0','C1','C2'])):
        ax.scatter(X2_iris[y_iris==cls,0], X2_iris[y_iris==cls,1],
                   label=name, s=20, color=col, edgecolors='k', linewidths=0.4)
    ax.set_xlabel('petal_length'); ax.set_ylabel('petal_width')
    ax.set_title(title); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

print(f'LDA accuracy: {lda2.score(X2_iris, y_iris):.4f}')
print(f'QDA accuracy: {qda2.score(X2_iris, y_iris):.4f}')

In [ ]:
# LDA vs QDA on Default: AUC comparison
lda_def = LinearDiscriminantAnalysis().fit(Xtr, ytr)
qda_def = QuadraticDiscriminantAnalysis().fit(Xtr, ytr)
lr_def  = LogisticRegression(max_iter=2000).fit(Xtr, ytr)

print('AUC comparison on Default (balance + income, held-out set):')
for name, model in [('LDA', lda_def), ('QDA', qda_def), ('LR', lr_def)]:
    auc_score = roc_auc_score(yte, model.predict_proba(Xte)[:, 1])
    acc_score = model.score(Xte, yte)
    print(f'  {name:5s}: AUC = {auc_score:.4f}, Accuracy = {acc_score:.4f}')

## Recap

1. **Generative vs discriminative**: LDA models class distributions; LR models the boundary.
2. **Bayes theorem**: Pr(Y=k|X=x) ∝ π_k · f_k(x). LDA approximates f_k with Gaussians.
3. **Shared Σ cancels the quadratic term** in the log-posterior → δ_k(x) linear in x → **linear boundary**.
4. **δ_k(x) = xᵀΣ⁻¹μ_k − ½μ_kᵀΣ⁻¹μ_k + log π_k** — the LDA discriminant score.
5. **Fisher projection**: project to min(K−1, p) axes for the best separation; same direction as LDA.
6. **QDA = separate Σ_k**: quadratic boundary, more flexible, needs more data.
7. **LDA AUC ≈ LR AUC** on Default — when Gaussian assumptions approximately hold, both agree.

---

# Exercises

## Exercise 1

**Task 1.** Fit LDA, QDA, and LR on the Default dataset using `balance`, `income`, and the binary `student` indicator.
Compare AUC on a held-out set. Which method wins and why?

In [ ]:
# your code here

### Exercise 1 — Solution

In [ ]:
d['student_num'] = (d['student'] == 'Yes').astype(float)
X3 = d[['balance', 'income', 'student_num']].to_numpy()
Xtr3, Xte3, ytr3, yte3 = train_test_split(X3, y_bin, test_size=0.3, random_state=0, stratify=y_bin)
print('AUC on Default (balance + income + student):')
for name, Model in [('LDA', LinearDiscriminantAnalysis),
                    ('QDA', QuadraticDiscriminantAnalysis),
                    ('LR',  lambda: LogisticRegression(max_iter=2000))]:
    m = (Model() if callable(Model) else Model()).fit(Xtr3, ytr3)
    print(f'  {name}: AUC = {roc_auc_score(yte3, m.predict_proba(Xte3)[:,1]):.4f}')

## Exercise 2

**Task 2.** Compute the Fisher LDA projection on Iris **by hand** (using numpy only — no sklearn LDA).
Plot the 1D histogram of LD1 scores per class.

In [ ]:
# your code here

### Exercise 2 — Solution

In [ ]:
# Binary Fisher LDA: class 0 vs 1 (setosa vs versicolor)
mask01 = y_iris < 2
X_bin = X4[mask01]; y_bin_iris = y_iris[mask01]
mu0 = X_bin[y_bin_iris == 0].mean(axis=0)
mu1 = X_bin[y_bin_iris == 1].mean(axis=0)
S0 = (X_bin[y_bin_iris==0] - mu0).T @ (X_bin[y_bin_iris==0] - mu0)
S1 = (X_bin[y_bin_iris==1] - mu1).T @ (X_bin[y_bin_iris==1] - mu1)
SW = S0 + S1
w = np.linalg.solve(SW, mu1 - mu0)  # Fisher direction
w /= np.linalg.norm(w)              # normalize
scores = X_bin @ w

fig, ax = plt.subplots(figsize=(7, 4))
for cls, name, col in [(0, 'setosa', 'C0'), (1, 'versicolor', 'C1')]:
    ax.hist(scores[y_bin_iris == cls], bins=15, alpha=0.6, label=name, color=col)
ax.set_xlabel('Fisher LD1 score'); ax.set_ylabel('count')
ax.set_title('Fisher projection (numpy) — setosa vs versicolor')
ax.legend(); plt.show()

## Exercise 3

**Task 3.** Load or simulate the South African Heart Disease dataset (or use Iris with all 3 classes).
Fit LDA with 4 features and plot the 2D Fisher projection (LD1 vs LD2) colored by class.

In [ ]:
# your code here

### Exercise 3 — Solution

In [ ]:
# Use Iris all 3 classes (4 features → 2 discriminant axes)
lda_full = LinearDiscriminantAnalysis().fit(X4, y_iris)
X_ld = lda_full.transform(X4)
print(f'Explained variance: LD1={lda_full.explained_variance_ratio_[0]:.3f}, '
      f'LD2={lda_full.explained_variance_ratio_[1]:.3f}')
fig, ax = plt.subplots(figsize=(7, 5))
for cls, (name, col) in enumerate(zip(le.classes_, ['C0','C1','C2'])):
    m = y_iris == cls
    ax.scatter(X_ld[m,0], X_ld[m,1], label=name, color=col, s=30,
               edgecolors='k', linewidths=0.4)
ax.set_xlabel('LD1'); ax.set_ylabel('LD2')
ax.set_title('Fisher (LDA) 2D projection — Iris 4 features → 2 axes')
ax.legend(); plt.show()